<a href="https://colab.research.google.com/github/onur34/XLM-RoBERTa-Sentiment-Analysis/blob/main/01_Model_Egitimi_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
import pandas as pd

tum_veriler = []

# 1. Mağaza Yorumları
try:
    df1 = pd.read_csv("magaza_yorumlari_duygu_analizi.csv", encoding='utf-16')
except:
    df1 = pd.read_csv("magaza_yorumlari_duygu_analizi.csv", encoding='windows-1254')
df1 = df1.rename(columns={"Görüş": "metin", "Durum": "etiket"})
df1 = df1[df1['etiket'].isin(['Olumlu', 'Olumsuz'])]
df1['etiket'] = df1['etiket'].map({'Olumlu': 1, 'Olumsuz': 0})
tum_veriler.append(df1[['metin', 'etiket']].dropna())

# 2. Tweetler
df2 = pd.read_csv("Tweets.csv")
df2 = df2.rename(columns={"text": "metin", "sentiment": "etiket"})
df2 = df2[df2['etiket'].isin(['positive', 'negative'])]
df2['etiket'] = df2['etiket'].map({'positive': 1, 'negative': 0})
tum_veriler.append(df2[['metin', 'etiket']].dropna().sample(10000, random_state=42))

# 3. Kadın Giyim (E-Ticaret)
df3 = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")
df3 = df3.rename(columns={"Review Text": "metin", "Rating": "etiket"})
df3 = df3[df3['etiket'] != 3]
df3['etiket'] = df3['etiket'].apply(lambda x: 1 if float(x) > 3 else 0)
tum_veriler.append(df3[['metin', 'etiket']].dropna().sample(10000, random_state=42))

# 4. Sentiment Dataset
df4 = pd.read_csv("sentimentdataset.csv")
df4 = df4.rename(columns={"Text": "metin", "Sentiment": "etiket"})
df4['etiket'] = df4['etiket'].str.strip().str.lower()
df4 = df4[df4['etiket'].isin(['positive', 'negative'])]
df4['etiket'] = df4['etiket'].map({'positive': 1, 'negative': 0})
tum_veriler.append(df4[['metin', 'etiket']].dropna())

# 5. Türk Filmleri (Eski Verimiz)
try:
    df5 = pd.read_csv("turkish_movie_sentiment_dataset.csv", encoding='utf-8')
except:
    df5 = pd.read_csv("turkish_movie_sentiment_dataset.csv", encoding='windows-1254')
if 'comment' in df5.columns:
    df5 = df5.rename(columns={"comment": "metin"})
if 'point' in df5.columns:
    df5['etiket'] = df5['point'].astype(str).str.replace(',', '.').astype(float)
    df5['etiket'] = df5['etiket'].apply(lambda x: 1 if x > 3 else 0)
tum_veriler.append(df5[['metin', 'etiket']].dropna())

# 6. IMDB Filmleri (Eski Verimiz)
df6 = pd.read_csv("IMDB Dataset.csv")
df6 = df6.rename(columns={"review": "metin", "sentiment": "etiket"})
df6['etiket'] = df6['etiket'].map({'positive': 1, 'negative': 0})
tum_veriler.append(df6[['metin', 'etiket']].dropna().sample(10000, random_state=42))

# BİRLEŞTİRME VE EĞİTİM İÇİN SEÇME
dev_df = pd.concat(tum_veriler, ignore_index=True)
df_olumlu = dev_df[dev_df['etiket'] == 1].sample(10000, random_state=42)
df_olumsuz = dev_df[dev_df['etiket'] == 0].sample(10000, random_state=42)
df_final = pd.concat([df_olumlu, df_olumsuz]).sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv("dev_egitim_verisi_20k.csv", index=False)
print(f"Eğitime girecek nihai veri hazır: {len(df_final)} satır.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# Metrik Hesaplama
def metrikleri_hesapla(eval_pred):
    tahminler, gercek_degerler = eval_pred
    tahmin_edilen_siniflar = np.argmax(tahminler, axis=1)
    return {
        'accuracy': accuracy_score(gercek_degerler, tahmin_edilen_siniflar),
        'f1': f1_score(gercek_degerler, tahmin_edilen_siniflar),
        'precision': precision_score(gercek_degerler, tahmin_edilen_siniflar),
        'recall': recall_score(gercek_degerler, tahmin_edilen_siniflar)
    }

# Veriyi Hazırla
df = pd.read_csv("dev_egitim_verisi_20k.csv")
train_texts, test_texts, train_labels, test_labels = train_test_split(df['metin'].astype(str).tolist(), df['etiket'].tolist(), test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=2)

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

class DuyguDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = DuyguDataset(train_encodings, train_labels)
test_dataset = DuyguDataset(test_encodings, test_labels)

# Eğitim Ayarları
training_args = TrainingArguments(
    output_dir='./sonuclar',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=metrikleri_hesapla
)

print("EĞİTİM BAŞLIYOR...")
trainer.train()

# Eğitimi Biten Modeli Drive'a Kaydetme
kayit_klasoru = '/content/drive/MyDrive/YZ_Proje_Duygu_Analizi/03_Eğitim_Modeli'
trainer.save_model(kayit_klasoru)
tokenizer.save_pretrained(kayit_klasoru)
print("Model başarıyla Drive'a kaydedildi!")